# SLM-WM 真实科学算子与仅图像盲检入口

该 Notebook 是当前主方法的正式 GPU 入口。它运行分支风险、语义条件 Jacobian 低响应子空间、空间 LF 载体、幅值尾部载体、真实 Q/K 注意力几何、仅图像盲检、真实图像攻击和正式 Inception FID/KID。

长耗时结果保存在 Google Drive 中的专用 repository 工作区。Colab 中断后重新打开本 Notebook 并执行全部单元格, 程序会依据配置摘要、代码版本和 manifest 复用已完成 Prompt, 不会把半写入结果当作正式证据。

## 使用顺序

1. 首次直接使用默认 `probe_paper`, 完成 70 个 Prompt 的端到端验证。
2. 把 Colab 运行时设置为 GPU, 建议使用显存不低于 24GB 的 L4、A100 或同等级设备。
3. 在 Colab 密钥中配置 `HF_TOKEN`, 且账号已获得 SD3.5 Medium 权限。
4. `SLM_WM_MAX_NEW_PROMPTS_PER_SESSION` 控制单次会话新增 Prompt 数。会话结束或中断后可重复执行全部单元格。
5. 数据集主流程完成后, 再把 `SLM_WM_RUN_FORMAL_ABLATION` 改为 `True`。正式消融同样支持跨会话续跑。
6. `workflow_decision` 为 `resume_required` 只表示仍需续跑, 不表示实验失败。只有正式摘要和结果包齐备后才能进入论文结果闭合。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os
from pathlib import Path
import shutil

from google.colab import drive

drive.mount('/content/drive')
# 默认运行 probe_paper; 通过后再显式切换到 pilot_paper 或 full_paper.
SLM_WM_MAX_NEW_PROMPTS_PER_SESSION = 5
SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION = 5
SLM_WM_RUN_FORMAL_ABLATION = False

os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
os.environ["SLM_WM_MAX_NEW_PROMPTS_PER_SESSION"] = str(SLM_WM_MAX_NEW_PROMPTS_PER_SESSION)
os.environ["SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION"] = str(SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION)
os.environ.setdefault("SLM_WM_ENABLE_DIFFUSION_ATTACKS", "1")
os.environ.setdefault("SLM_WM_INCEPTION_DEVICE", "cuda")

# 模型缓存与运行输出都保存在 Drive, 以便 Colab 断开后继续。
drive_root = Path("/content/drive/MyDrive/SLM")
hf_home = drive_root / "model_cache" / "huggingface"
torch_home = drive_root / "model_cache" / "torch"
hf_home.mkdir(parents=True, exist_ok=True)
torch_home.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(hf_home))
os.environ.setdefault("TORCH_HOME", str(torch_home))
drive_usage = shutil.disk_usage(drive_root)
print({
    "paper_run_name": SLM_WM_PAPER_RUN_NAME,
    "hf_home": str(hf_home),
    "torch_home": str(torch_home),
    "drive_free_gib": round(drive_usage.free / 1024**3, 2),
})

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/drive/MyDrive/SLM/workspaces/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})

In [ ]:
DEPENDENCY_PROFILE_ID = "sd35_method_runtime_gpu"
dependency_profile_result = subprocess.run(
    ["python", "scripts/prepare_dependency_profile.py", "--profile", DEPENDENCY_PROFILE_ID],
    check=True,
)
print({"dependency_profile_id": DEPENDENCY_PROFILE_ID, "return_code": dependency_profile_result.returncode})

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report

dependency_report = build_notebook_dependency_report(DEPENDENCY_PROFILE_ID)
print(dependency_report)
assert dependency_report["dependency_decision"] == "pass", dependency_report


In [ ]:
try:
    from google.colab import userdata
    token_from_secret = userdata.get("HF_TOKEN")
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = token_from_secret
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("缺少 Colab 密钥 HF_TOKEN")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hugging Face 登录完成")

In [ ]:
from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment

paper_run_environment = configure_paper_run_environment(
    workflow_name="semantic_watermark_image_only",
)
print(paper_run_environment)


In [ ]:
import torch

device_report = {
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "total_memory_gib": (
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
        if torch.cuda.is_available()
        else 0.0
    ),
}
print(device_report)
assert device_report["cuda_available"], "该正式入口要求 Colab GPU 运行时"
assert device_report["total_memory_gib"] >= 20.0, "建议改用显存不低于 24GB 的 Colab GPU"

In [ ]:
from paper_workflow.colab_utils.semantic_watermark_image_only import (
    run_semantic_watermark_image_only_session,
)

session_summary = run_semantic_watermark_image_only_session(
    root=".",
    run_formal_ablation=SLM_WM_RUN_FORMAL_ABLATION,
)
session_summary

In [ ]:
import json

print(json.dumps(session_summary, ensure_ascii=False, sort_keys=True, indent=2))
if session_summary["workflow_decision"] == "resume_required":
    print("当前会话已安全保存进度。下次连接 Colab 后重新执行全部单元格。")
elif session_summary["workflow_decision"] == "dataset_complete":
    print("数据集主流程与正式 FID/KID 已完成。需要正式消融时, 把开关改为 True 后继续运行。")
else:
    print("当前请求的主流程和正式消融均已完成, 结果包已经镜像到论文运行目录。")

In [ ]:
# 仅列出受治理摘要和结果包路径, 不在 Notebook 内重写论文结论。
run_name = paper_run_environment["paper_run_name"]
status_paths = [
    Path(f"outputs/image_only_dataset_runtime/{run_name}/dataset_runtime_progress.json"),
    Path(f"outputs/image_only_dataset_runtime/{run_name}/dataset_runtime_summary.json"),
    Path(f"outputs/dataset_level_quality/{run_name}/dataset_quality_summary.json"),
    Path(f"outputs/formal_mechanism_ablation/{run_name}/runtime_rerun_progress.json"),
    Path(f"outputs/formal_mechanism_ablation/{run_name}/ablation_claim_summary.json"),
]
for status_path in status_paths:
    if status_path.is_file():
        print(f"\n{status_path}")
        print(status_path.read_text(encoding="utf-8")[:4000])
print({"drive_result_root": paper_run_environment["drive_result_root"]})